## Initialization

In [7]:
#@title
# ! git clone https://github.com/proroklab/VectorizedMultiAgentSimulator.git
# %cd /content/VectorizedMultiAgentSimulator
# !pip install -e .
!pip install "ray[rllib]"==2.2 wandb
!pip install "pydantic<2" numpy==1.23.5

I0000 00:00:1724438145.714478 2954911 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers


I0000 00:00:1724438146.760938 2954911 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers


In [8]:
#@title
# !sudo apt-get update
# !sudo apt-get install python3-opengl xvfb
# !pip install pyvirtualdisplay
import pyvirtualdisplay
display = pyvirtualdisplay.Display(visible=False, size=(1400, 900))
display.start()

I0000 00:00:1724438148.122169 2954911 fork_posix.cc:77] Other threads are currently calling into gRPC, skipping fork() handlers


## Run


In [9]:
import torch
import os
has_gpu = torch.cuda.is_available()
os.environ["RLLIB_NUM_GPUS"] = "1" if has_gpu else "0"

In [10]:
#  Copyright (c) 2022.
#  ProrokLab (https://www.proroklab.org/)
#  All rights reserved.

import os
from typing import Dict, Optional

import numpy as np
import ray
from ray import tune
from ray.rllib import BaseEnv, Policy, RolloutWorker
from ray.rllib.algorithms.callbacks import DefaultCallbacks
from ray.rllib.agents.ppo import PPOTrainer
from ray.rllib.evaluation import Episode, MultiAgentEpisode
from ray.rllib.utils.typing import PolicyID
from ray.tune import register_env

import wandb
from vmas import make_env, Wrapper

# scenario_name = "balance"
scenario_name = "navigation"

# Scenario specific variables.
# When modifying this also modify env_config and env_creator
n_agents = 4

# Common variables
continuous_actions = True
max_steps = 200
num_vectorized_envs = 96
num_workers = 1
vmas_device = "cuda"  # or cuda


def env_creator(config: Dict):
    env = make_env(
        scenario=config["scenario_name"],
        num_envs=config["num_envs"],
        device=config["device"],
        continuous_actions=config["continuous_actions"],
        wrapper=Wrapper.RLLIB,
        max_steps=config["max_steps"],
        # Scenario specific variables
        n_agents=config["n_agents"],
    )
    return env


if not ray.is_initialized():
    ray.init()
    print("Ray init!")
register_env(scenario_name, lambda config: env_creator(config))


class EvaluationCallbacks(DefaultCallbacks):
    def on_episode_step(
        self,
        *,
        worker: RolloutWorker,
        base_env: BaseEnv,
        episode: MultiAgentEpisode,
        **kwargs,
    ):
        info = episode.last_info_for()
        for a_key in info.keys():
            for b_key in info[a_key]:
                try:
                    episode.user_data[f"{a_key}/{b_key}"].append(info[a_key][b_key])
                except KeyError:
                    episode.user_data[f"{a_key}/{b_key}"] = [info[a_key][b_key]]

    def on_episode_end(
        self,
        *,
        worker: RolloutWorker,
        base_env: BaseEnv,
        policies: Dict[str, Policy],
        episode: MultiAgentEpisode,
        **kwargs,
    ):
        info = episode.last_info_for()
        for a_key in info.keys():
            for b_key in info[a_key]:
                metric = np.array(episode.user_data[f"{a_key}/{b_key}"])
                episode.custom_metrics[f"{a_key}/{b_key}"] = np.sum(metric).item()


class RenderingCallbacks(DefaultCallbacks):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.frames = []

    def on_episode_step(
        self,
        *,
        worker: RolloutWorker,
        base_env: BaseEnv,
        policies: Optional[Dict[PolicyID, Policy]] = None,
        episode: Episode,
        **kwargs,
    ) -> None:
        self.frames.append(base_env.vector_env.try_render_at(mode="rgb_array"))

    def on_episode_end(
        self,
        *,
        worker: RolloutWorker,
        base_env: BaseEnv,
        policies: Dict[PolicyID, Policy],
        episode: Episode,
        **kwargs,
    ) -> None:
        vid = np.transpose(self.frames, (0, 3, 1, 2))
        episode.media["rendering"] = wandb.Video(vid, fps=1 / base_env.vector_env.env.world.dt, format="mp4")
        self.frames = []


def train():

    RLLIB_NUM_GPUS = int(os.environ.get("RLLIB_NUM_GPUS", "0"))
    num_gpus = 0.001 if RLLIB_NUM_GPUS > 0 else 0  # Driver GPU
    num_gpus_per_worker = (
        (RLLIB_NUM_GPUS - num_gpus) / (num_workers + 1) if vmas_device == "cuda" else 0
    )

    tune.run(
        PPOTrainer,
        stop={"training_iteration": 200},
        checkpoint_freq=1,
        keep_checkpoints_num=2,
        checkpoint_at_end=True,
        checkpoint_score_attr="episode_reward_mean",
        # callbacks=[
        #     WandbLoggerCallback(
        #        project=f"{scenario_name}",
        #        api_key="",
        #    )
        # ],
        config={
            "seed": 0,
            "framework": "torch",
            "env": scenario_name,
            "kl_coeff": 0.01,
            "kl_target": 0.01,
            "lambda": 0.9,
            "clip_param": 0.2,
            "vf_loss_coeff": 1,
            "vf_clip_param": float("inf"),
            "entropy_coeff": 0,
            "train_batch_size": 60000,
            "rollout_fragment_length": 125,
            "sgd_minibatch_size": 4096,
            "num_sgd_iter": 40,
            "num_gpus": num_gpus,
            "num_workers": num_workers,
            "num_gpus_per_worker": num_gpus_per_worker,
            "num_envs_per_worker": num_vectorized_envs,
            "lr": 5e-5,
            "gamma": 0.99,
            "use_gae": True,
            "use_critic": True,
            "batch_mode": "truncate_episodes",
            "env_config": {
                "device": vmas_device,
                "num_envs": num_vectorized_envs,
                "scenario_name": scenario_name,
                "continuous_actions": continuous_actions,
                "max_steps": max_steps,
                # Scenario specific variables
                "n_agents": n_agents,
            },
            "evaluation_interval": 5,
            "evaluation_duration": 1,
            "evaluation_num_workers": 0,
            "evaluation_parallel_to_training": False,
            "evaluation_config": {
                "num_envs_per_worker": 1,
                "env_config": {
                    "num_envs": 1,
                },
                # "callbacks": MultiCallbacks([RenderingCallbacks, EvaluationCallbacks]),
            },
            "callbacks": EvaluationCallbacks,
        },
    )


if __name__ == "__main__":
    train()

Trial name,agent_timesteps_total,counters,custom_metrics,date,done,episode_len_mean,episode_media,episode_reward_max,episode_reward_mean,episode_reward_min,episodes_this_iter,episodes_total,evaluation,experiment_id,hostname,info,iterations_since_restore,node_ip,num_agent_steps_sampled,num_agent_steps_trained,num_env_steps_sampled,num_env_steps_sampled_this_iter,num_env_steps_trained,num_env_steps_trained_this_iter,num_faulty_episodes,num_healthy_workers,num_in_flight_async_reqs,num_remote_worker_restarts,num_steps_trained_this_iter,perf,pid,policy_reward_max,policy_reward_mean,policy_reward_min,sampler_perf,time_since_restore,time_this_iter_s,time_total_s,timers,timestamp,timesteps_since_restore,timesteps_total,training_iteration,trial_id,warmup_time
PPO_navigation_84230_00000,12000000,"{'num_env_steps_sampled': 12000000, 'num_env_steps_trained': 12000000, 'num_agent_steps_sampled': 12000000, 'num_agent_steps_trained': 12000000}","{'rewards/0_mean': 4.1156544986591275, 'rewards/0_min': 0.917890053242445, 'rewards/0_max': 7.323945930227637, 'rewards/1_mean': 4.127009532915993, 'rewards/1_min': 0.35848845168948174, 'rewards/1_max': 7.323945930227637, 'rewards/2_mean': 4.112626489603302, 'rewards/2_min': 0.4701324962079525, 'rewards/2_max': 7.323945930227637, 'rewards/3_mean': 4.107327473649852, 'rewards/3_min': 0.35848845168948174, 'rewards/3_max': 7.323945930227637, 'agent_0/pos_rew_mean': 4.161831637407688, 'agent_0/pos_rew_min': 1.7416461780667305, 'agent_0/pos_rew_max': 7.323945930227637, 'agent_0/final_rew_mean': 0.0, 'agent_0/final_rew_min': 0.0, 'agent_0/final_rew_max': 0.0, 'agent_0/agent_collisions_mean': -0.04617713853141559, 'agent_0/agent_collisions_min': -4.0, 'agent_0/agent_collisions_max': 0.0, 'agent_1/pos_rew_mean': 4.161831637407688, 'agent_1/pos_rew_min': 1.7416461780667305, 'agent_1/pos_rew_max': 7.323945930227637, 'agent_1/final_rew_mean': 0.0, 'agent_1/final_rew_min': 0.0, 'agent_1/final_rew_max': 0.0, 'agent_1/agent_collisions_mean': -0.0348221044663134, 'agent_1/agent_collisions_min': -3.0, 'agent_1/agent_collisions_max': 0.0, 'agent_2/pos_rew_mean': 4.161831637407688, 'agent_2/pos_rew_min': 1.7416461780667305, 'agent_2/pos_rew_max': 7.323945930227637, 'agent_2/final_rew_mean': 0.0, 'agent_2/final_rew_min': 0.0, 'agent_2/final_rew_max': 0.0, 'agent_2/agent_collisions_mean': -0.04920514761544285, 'agent_2/agent_collisions_min': -3.0, 'agent_2/agent_collisions_max': 0.0, 'agent_3/pos_rew_mean': 4.161831637407688, 'agent_3/pos_rew_min': 1.7416461780667305, 'agent_3/pos_rew_max': 7.323945930227637, 'agent_3/final_rew_mean': 0.0, 'agent_3/final_rew_min': 0.0, 'agent_3/final_rew_max': 0.0, 'agent_3/agent_collisions_mean': -0.05450416351249054, 'agent_3/agent_collisions_min': -4.0, 'agent_3/agent_collisions_max': 0.0}",2024-08-23_17-14-31,True,45.7282,{},7.32395,4.11565,1.47013,1321,173880,"{'episode_reward_max': 4.134781688451767, 'episode_reward_min': 4.134781688451767, 'episode_reward_mean': 4.134781688451767, 'episode_len_mean': 37.0, 'episode_media': {}, 'episodes_this_iter': 1, 'policy_reward_min': {}, 'policy_reward_max': {}, 'policy_reward_mean': {}, 'custom_metrics': {'rewards/0_mean': 4.134781688451767, 'rewards/0_min': 4.134781688451767, 'rewards/0_max': 4.134781688451767, 'rewards/1_mean': 4.134781688451767, 'rewards/1_min': 4.134781688451767, 'rewards/1_max': 4.134781688451767, 'rewards/2_mean': 4.134781688451767, 'rewards/2_min': 4.134781688451767, 'rewards/2_max': 4.134781688451767, 'rewards/3_mean': 4.134781688451767, 'rewards/3_min': 4.134781688451767, 'rewards/3_max': 4.134781688451767, 'agent_0/pos_rew_mean': 4.134781688451767, 'agent_0/pos_rew_min': 4.134781688451767, 'agent_0/pos_rew_max': 4.134781688451767, 'agent_0/final_rew_mean': 0.0, 'agent_0/final_rew_min': 0.0, 'agent_0/final_rew_max': 0.0, 'agent_0/agent_collisions_mean': 0.0, 'agent_0/agent_collisions_min': 0.0, 'agent_0/agent_collisions_max': 0.0, 'agent_1/pos_rew_mean': 4.134781688451767, 'agent_1/pos_rew_min': 4.134781688451767, 

2024-08-23 17:14:32,089	INFO tune.py:762 -- Total run time: 9523.85 seconds (9523.38 seconds for the tuning loop).
